# Real Madrid AI Transfer Intelligence
## Notebook 06 — External Scouting Pool & Candidate Shortlist

**Objective:** expand the player database beyond Real Madrid and create a comparable external scouting pool for the 2025/26 completed season. This is the bridge between internal squad analysis and transfer-market scouting.

**Input:** FBref player-season tables collected through `soccerdata`.

**Scope:** La Liga, Premier League, Bundesliga, Serie A, Ligue 1, Primeira Liga, Eredivisie and Belgian Pro League, season 2025/26. These are the eight European leagues selected for this project. The exact competition coverage returned by the source must be checked in the output.

**Important**
- This notebook creates a statistical scouting pool, not a final transfer recommendation.
- It does not invent market values, contract details, release clauses or transfer fees.
- Candidate similarity is position-specific and based on available performance statistics.
- The 2025/26 season is historical evidence. It does not establish current 2026/27 form, availability or price.
- Yan Diomande is handled as a normal player record in the unified player database. Do not add him manually as a Real Madrid player in 2025/26 unless the source confirms that club-season record.


## 1. Setup and source configuration

In [6]:
from pathlib import Path
import re, json, unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in [RAW_DIR, DATA_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SEASON = "2025-2026"

LEAGUES = [
    "ENG-Premier League",
    "ESP-La Liga",
    "GER-Bundesliga",
    "ITA-Serie A",
    "FRA-Ligue 1"
]

SOURCE = "FBref via SoccerData"
RETRIEVED_AT = datetime.now(timezone.utc).isoformat()

print("Season:", SEASON)
print("Leagues:", LEAGUES)
print("Retrieved at:", RETRIEVED_AT)


## 2. Download external player-season statistics

In [7]:
# Requires: pip install soccerdata pyarrow
import soccerdata as sd

fbref_external = sd.FBref(
    leagues=LEAGUES,
    seasons=[SEASON],
    data_dir=RAW_DIR / "fbref_external_cache"
)

STAT_TYPES = ["standard", "shooting", "playing_time", "misc"]
raw_tables = {}

for stat_type in STAT_TYPES:
    print(f"Downloading {stat_type}...")
    table = fbref_external.read_player_season_stats(stat_type=stat_type)
    raw_tables[stat_type] = table.copy()
    table.to_parquet(RAW_DIR / f"external_{SEASON}_{stat_type}.parquet")
    print("Rows:", len(table), "| Columns:", len(table.columns))


## 3. Flatten source tables and inspect schema

In [8]:
def flatten_source_table(table):
    out = table.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = [
            "_".join(str(x) for x in col if str(x) not in ("", "None"))
            for col in out.columns.to_list()
        ]
    else:
        out.columns = [str(c) for c in out.columns]

    if isinstance(out.index, pd.MultiIndex) or out.index.name is not None:
        out = out.reset_index()

    out.columns = [
        re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
        for c in out.columns
    ]
    return out

tables = {name: flatten_source_table(table) for name, table in raw_tables.items()}

for name, table in tables.items():
    print("\n", name, table.shape)
    print(table.columns.tolist())
    display(table.head(3))


,league,season,team,player,nation,pos,age,born,90s,performance_crdy,...,performance_2crdy,performance_fls,performance_fld,performance_off,performance_crs,performance_int,performance_tklw,performance_pkwon,performance_pkcon,performance_og
0,ENG-Premier League,2526,Arsenal,Ben White,ENG,DF,27,1997,7.8,0,...,0,5,6,3,16,9,9,<NA>,<NA>,0
1,ENG-Premier League,2526,Arsenal,Bukayo Saka,ENG,"FW,MF",23,2001,24.7,2,...,0,33,54,14,122,16,19,<NA>,<NA>,0
2,ENG-Premier League,2526,Arsenal,Christian Nørgaard,DEN,MF,31,1994,1.2,0,...,0,3,0,0,1,1,2,<NA>,<NA>,0


## 4. Standardize and merge metric tables

In [9]:
def find_column(frame, options):
    return next((c for c in options if c in frame.columns), None)

base = tables["standard"].copy()

# Find player/team/league/season identifiers.
identifier_candidates = {
    "player_name": ["player", "player_name"],
    "team": ["team", "squad"],
    "league": ["league", "comp", "competition"],
    "season": ["season"],
}
id_map = {k: find_column(base, v) for k, v in identifier_candidates.items()}
display(pd.DataFrame([{"field": k, "source_column": v} for k, v in id_map.items()]))

if id_map["player_name"] is None:
    raise ValueError(f"Player name column not found. Columns: {base.columns.tolist()}")

# Keep one canonical base row per player-team-season-league.
for canonical, source_col in id_map.items():
    if source_col and canonical not in base.columns:
        base[canonical] = base[source_col]

join_keys = [c for c in ["player_name", "team", "league", "season"] if c in base.columns]
if not {"player_name", "team"}.issubset(join_keys):
    raise ValueError(f"Insufficient join keys: {join_keys}")

merged = base.copy()
merge_audit = []

for stat_type in ["shooting", "playing_time", "misc"]:
    supplement = tables[stat_type].copy()
    for canonical, source_col in id_map.items():
        if source_col and canonical not in supplement.columns and source_col in supplement.columns:
            supplement[canonical] = supplement[source_col]

    keys = [k for k in join_keys if k in supplement.columns]
    if len(keys) < 2 or "player_name" not in keys:
        merge_audit.append({"table": stat_type, "status": "skipped", "reason": f"insufficient keys {keys}"})
        continue

    # Select only new metric columns to avoid duplicate field names.
    new_cols = [c for c in supplement.columns if c not in merged.columns or c in keys]
    supplement = supplement[new_cols].copy()

    duplicate_keys = int(supplement.duplicated(keys).sum())
    if duplicate_keys:
        merge_audit.append({"table": stat_type, "status": "skipped", "reason": f"{duplicate_keys} duplicate join keys"})
        continue

    merged = merged.merge(supplement, on=keys, how="left", validate="one_to_one")
    merge_audit.append({"table": stat_type, "status": "merged", "reason": "", "keys": keys})

display(pd.DataFrame(merge_audit))
print("Merged shape:", merged.shape)


## 5. Create unified player-season schema

In [11]:
# Resolve source naming variants to canonical fields.
canonical_candidates = {
    "minutes": ["minutes", "playing_time_min", "min"],
    "matches": ["matches", "playing_time_mp", "mp"],
    "starts": ["starts", "playing_time_starts"],
    "goals": ["goals", "performance_gls", "standard_gls", "gls"],
    "assists": ["assists", "performance_ast", "standard_ast", "ast"],
    "position": ["position", "pos"],
    "age": ["age"],
    "goals_per_90": ["goals_per_90", "per_90_minutes_gls"],
    "assists_per_90": ["assists_per_90", "per_90_minutes_ast"],
    "goal_contributions_per_90": ["goal_contributions_per_90", "per_90_minutes_g_a"],
    "shots_per_90": ["shots_per_90", "standard_sh_90"],
    "shots_on_target_per_90": ["shots_on_target_per_90", "standard_sot_90"],
    "tackles_won": ["performance_tklw", "tackles_won"],
    "interceptions": ["performance_int", "interceptions"],
    "crosses": ["performance_crs", "crosses"],
}
for canonical, options in canonical_candidates.items():
    source_col = find_column(merged, options)
    if source_col is not None and canonical not in merged.columns:
        merged[canonical] = merged[source_col]

for col in ["minutes", "matches", "starts", "goals", "assists", "age",
            "goals_per_90", "assists_per_90", "goal_contributions_per_90",
            "shots_per_90", "shots_on_target_per_90", "tackles_won",
            "interceptions", "crosses"]:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors="coerce")

if "minutes" not in merged.columns:
    raise ValueError("Minutes field not found; inspect the source schema.")

# Derive rate metrics from counts only when source per-90 values are unavailable.
for count_col, rate_col in [
    ("goals", "goals_per_90"),
    ("assists", "assists_per_90"),
    ("tackles_won", "tackles_won_per_90"),
    ("interceptions", "interceptions_per_90"),
    ("crosses", "crosses_per_90"),
]:
    if rate_col not in merged.columns and count_col in merged.columns:
        merged[rate_col] = np.where(
            merged["minutes"] > 0,
            merged[count_col] * 90 / merged["minutes"],
            np.nan
        )

def normalize_position(value):
    if pd.isna(value):
        return "Unknown"
    tokens = set(re.split(r"[,;/ ]+", str(value).upper()))
    if "GK" in tokens: return "GK"
    if "FW" in tokens: return "FW"
    if "MF" in tokens: return "MF"
    if "DF" in tokens: return "DF"
    return "Unknown"

merged["position_group"] = merged["position"].map(normalize_position) if "position" in merged.columns else "Unknown"

def player_id(name):
    if pd.isna(name):
        return pd.NA
    value = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")

merged["player_id"] = merged["player_name"].map(player_id)
merged["source"] = SOURCE
merged["retrieved_at_utc"] = RETRIEVED_AT
merged["scope"] = "External scouting pool; five FBref-supported leagues; 2025/26"

display(merged[["player_name", "team", "league", "season", "position_group", "minutes"]].head(15))



,player_name,team,league,season,position_group,minutes
0,Ben White,Arsenal,ENG-Premier League,2526,DF,702
1,Bukayo Saka,Arsenal,ENG-Premier League,2526,FW,2222
2,Christian Nørgaard,Arsenal,ENG-Premier League,2526,MF,106
3,Cristhian Mosquera,Arsenal,ENG-Premier League,2526,DF,994
4,David Raya,Arsenal,ENG-Premier League,2526,GK,3330
5,Declan Rice,Arsenal,ENG-Premier League,2526,MF,3094
6,Eberechi Eze,Arsenal,ENG-Premier League,2526,FW,1812
7,Ethan Nwaneri,Arsenal,ENG-Premier League,2526,MF,171
8,Gabriel Jesus,Arsenal,ENG-Premier League,2526,FW,429
9,Gabriel Magalhães,Arsenal,ENG-Premier League,2526,DF,2751


In [12]:
# Preserve full raw scouting pool; create a separate eligible shortlist pool.
quality = {
    "rows": int(len(merged)),
    "unique_players": int(merged["player_id"].nunique()),
    "unique_teams": int(merged["team"].nunique()) if "team" in merged.columns else None,
    "leagues_returned": sorted(merged["league"].dropna().astype(str).unique().tolist()) if "league" in merged.columns else [],
    "missing_minutes": int(merged["minutes"].isna().sum()),
    "negative_minutes": int((merged["minutes"] < 0).sum()),
    "duplicate_player_team_rows": int(merged.duplicated([c for c in ["player_id", "team", "season"] if c in merged.columns]).sum())
}
display(pd.DataFrame([quality]))

MIN_MINUTES = 300  # wider discovery pool; low-minute samples remain flagged
pool = merged[
    merged["minutes"].ge(MIN_MINUTES)
    & merged["position_group"].isin(["FW", "MF", "DF"])
].copy()

print("Full pool:", len(merged), "| Eligible outfield pool:", len(pool))
display(pool.groupby(["league", "position_group"]).size().rename("player_rows").reset_index())


,league,position_group,player_rows
0,ENG-Premier League,DF,117
1,ENG-Premier League,FW,97
2,ENG-Premier League,MF,184
3,ESP-La Liga,DF,136
4,ESP-La Liga,FW,121
5,ESP-La Liga,MF,168
6,FRA-Ligue 1,DF,94
7,FRA-Ligue 1,FW,108
8,FRA-Ligue 1,MF,177
9,GER-Bundesliga,DF,92


## 7. Feature coverage and comparable player profiles

In [13]:
SCOUTING_FEATURES = [
    "goals_per_90", "assists_per_90", "goal_contributions_per_90",
    "shots_per_90", "shots_on_target_per_90",
    "tackles_won_per_90", "interceptions_per_90", "crosses_per_90"
]
scouting_features = [
    c for c in SCOUTING_FEATURES
    if c in pool.columns and pool[c].notna().sum() >= 5 and pool[c].nunique(dropna=True) > 1
]

coverage = pd.DataFrame([
    {
        "feature": c,
        "non_missing": int(pool[c].notna().sum()),
        "missing_percent": round(pool[c].isna().mean() * 100, 1),
        "unique_values": int(pool[c].nunique(dropna=True))
    }
    for c in scouting_features
])
display(coverage)
print("Scouting features:", scouting_features)

if len(scouting_features) < 2:
    raise ValueError(
        "Not enough comparable features. Inspect source columns and metric merge audit."
    )


## 8. Create role-specific candidate shortlists

In [14]:
# This is a transparent screening score, not a learned transfer-success probability.
# We rank players within position group using percentile ranks of available features.
# The score is deliberately labelled a scouting screen, not a definitive quality score.

ROLE_WEIGHTS = {
    "FW": {
        "goals_per_90": 0.30, "assists_per_90": 0.15,
        "goal_contributions_per_90": 0.25, "shots_per_90": 0.15,
        "shots_on_target_per_90": 0.15
    },
    "MF": {
        "assists_per_90": 0.20, "goal_contributions_per_90": 0.15,
        "tackles_won_per_90": 0.20, "interceptions_per_90": 0.20,
        "crosses_per_90": 0.10, "goals_per_90": 0.15
    },
    "DF": {
        "tackles_won_per_90": 0.35, "interceptions_per_90": 0.30,
        "crosses_per_90": 0.15, "assists_per_90": 0.10,
        "goal_contributions_per_90": 0.10
    }
}

candidate_parts = []
for position, group in pool.groupby("position_group"):
    configured = {
        metric: weight for metric, weight in ROLE_WEIGHTS.get(position, {}).items()
        if metric in scouting_features
    }
    if not configured:
        continue

    weight_total = sum(configured.values())
    normalized_weights = {m: w / weight_total for m, w in configured.items()}

    ranks = pd.DataFrame({
        metric: group[metric].rank(pct=True) * 100
        for metric in normalized_weights
    }, index=group.index)

    weighted = pd.DataFrame({
        metric: ranks[metric] * weight
        for metric, weight in normalized_weights.items()
    })
    denom = pd.DataFrame({
        metric: ranks[metric].notna().astype(float) * weight
        for metric, weight in normalized_weights.items()
    }).sum(axis=1)

    group = group.copy()
    group["scouting_screen_score"] = weighted.sum(axis=1, min_count=1) / denom.replace(0, np.nan)
    group["scouting_metrics_used"] = ranks.notna().sum(axis=1)
    candidate_parts.append(group)

if not candidate_parts:
    raise ValueError("No candidate scores calculated. Check role weights and feature availability.")

candidates = pd.concat(candidate_parts, ignore_index=True)
candidates = candidates.sort_values(
    ["position_group", "scouting_screen_score"],
    ascending=[True, False]
)

shortlist = (
    candidates.groupby("position_group", group_keys=False)
    .head(20)  # target: up to 20 candidates per position (60 total)
    .copy()
)

display(shortlist[[
    "player_name", "team", "league", "position_group", "age", "minutes",
    "goals_per_90", "assists_per_90", "scouting_screen_score"
]].head(60))


,player_name,team,league,position_group,age,minutes,goals_per_90,assists_per_90,scouting_screen_score
413,Max Finkgräfe,RB Leipzig,GER-Bundesliga,DF,21,504,0.18,0.18,93.14882
246,Alfonso Pedraza,Villarreal,ESP-La Liga,DF,29,1689,0.05,0.16,91.669691
34,Quilindschy Hartman,Burnley,ENG-Premier League,DF,23,1825,0.0,0.25,91.306715
252,Sergi Cardona,Villarreal,ESP-La Liga,DF,26,1970,0.05,0.18,90.22686
376,Lukas Kübler,Freiburg,GER-Bundesliga,DF,32,1587,0.11,0.11,88.157895
267,Daouda Guindo,Brest,FRA-Ligue 1,DF,22,1686,0.05,0.05,85.181488
92,Neco Williams,Nottingham,ENG-Premier League,DF,24,3204,0.06,0.08,84.38294
199,Lucas Ahijado,Oviedo,ESP-La Liga,DF,30,1098,0.0,0.08,83.920145
320,Hamari Traoré,Paris FC,FRA-Ligue 1,DF,33,1150,0.0,0.08,83.865699
274,Enzo Koffi Vinette,Le Havre,FRA-Ligue 1,DF,19,385,0.0,0.0,83.720508


## 9. Export scouting pool, shortlist and source manifest

In [15]:
pool_path = DATA_DIR / "external_scouting_pool_2025_26.csv"
shortlist_path = DATA_DIR / "external_scouting_shortlist_2025_26.csv"
coverage_path = REPORTS_DIR / "external_scouting_feature_coverage.csv"
merge_audit_path = REPORTS_DIR / "external_scouting_merge_audit.csv"
manifest_path = REPORTS_DIR / "external_scouting_manifest.json"

merged.to_csv(pool_path, index=False)
shortlist.to_csv(shortlist_path, index=False)
coverage.to_csv(coverage_path, index=False)
pd.DataFrame(merge_audit).to_csv(merge_audit_path, index=False)

manifest = {
    "source": SOURCE,
    "season": SEASON,
    "leagues_requested": LEAGUES,
    "leagues_returned": quality["leagues_returned"],
    "retrieved_at_utc": RETRIEVED_AT,
    "minimum_minutes": MIN_MINUTES,
    "rows_full_pool": int(len(merged)),
    "rows_eligible_outfield_pool": int(len(pool)),
    "rows_shortlist": int(len(shortlist)),
    "features_used": scouting_features,
    "metric_merge_audit": merge_audit,
    "limitations": [
        "2025/26 historical statistics do not establish current 2026/27 form.",
        "No transfer fees, market values, contract status or availability are included.",
        "The shortlist targets up to 20 players per outfield position; actual counts depend on source coverage and the 300-minute eligibility filter.",
        "Players with 300–449 minutes have smaller samples and should be reviewed cautiously.",
        "Player roles are approximated from broad FBref position labels.",
        "League strength and tactical context differ across competitions.",
        "The screening score is descriptive and is not a probability of transfer success."
    ]
}
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print("Saved:", pool_path)
print("Saved:", shortlist_path)
print("Saved:", coverage_path)
print("Saved:", manifest_path)
